In [ ]:
from google.colab import files
upload=files.upload()

Saving twitter_training.csv to twitter_training (4).csv


In [ ]:
import pandas as pd
df=pd.read_csv("twitter_training.csv")
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [ ]:
df.groupby("Positive").count()

,2401,Borderlands,"im getting on borderlands and i will murder you all ,"
Positive,,,
Irrelevant,12990,12990,12875
Negative,22542,22542,22358
Neutral,18318,18318,18108
Positive,20831,20831,20654


In [ ]:
df = pd.read_csv("twitter_training.csv", encoding="utf-8", header=None)
df.columns = ["ID", "Category", "Sentiment", "Text"]
print(df.head())
df = df.drop(columns=["ID", "Category"])
df.info()
print(df["Sentiment"].isna().sum())
print(df["Text"].isna().sum())
print(df["Sentiment"].value_counts())

df[df["Sentiment"].notna()]
df[df["Text"].notna()]
print(df["Sentiment"].isna().sum())
print(df.shape)


     ID     Category Sentiment  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

                                                Text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Sentiment  74682 non-null  object
 1   Text       73996 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB
0
686
Sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64
0
(74682, 2)


In [ ]:
df.shape

(74682, 2)

In [ ]:
!pip install torch==2.1.2
!pip install torchtext==0.16.2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import pandas as pd
import re

In [ ]:
df.head()

,Sentiment,Text
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,I am coming to the borders and I will kill you...
2,Positive,im getting on borderlands and i will kill you ...
3,Positive,im coming on borderlands and i will murder you...
4,Positive,im getting on borderlands 2 and i will murder ...


In [ ]:
df["Text"] = df["Text"].fillna("")

tokenizer = get_tokenizer("basic_english")

def clean_and_tokenize(line):
    if not isinstance(line, str):
        return []
    line = re.sub(r'\d+', '', line)
    return tokenizer(line.lower())


def yield_tokens(text_iter):
    for line in text_iter:
        yield clean_and_tokenize(line)

vocab = build_vocab_from_iterator(yield_tokens(df["Text"]), specials=["<unk>", "<pad>"])
vocab.set_default_index(vocab["<unk>"])


text_pipeline = lambda x: vocab(clean_and_tokenize(x))


In [ ]:
print(vocab["love"])

68


In [ ]:
label_dict = {"Positive": 0, "Negative": 1, "Neutral": 2, "Irrelevant": 3}

data = list(zip(df["Text"], df["Sentiment"]))


def collate_batch(batch):
    label_list, text_list = [], []
    for text, label in batch:
        label_list.append(label_dict[label])
        processed_text = torch.tensor(text_pipeline(text), dtype=torch.int64)
        text_list.append(processed_text)
    text_tensor = pad_sequence(text_list, batch_first=True, padding_value=vocab["<pad>"])
    label_tensor = torch.tensor(label_list, dtype=torch.int64)
    return text_tensor.to(device), label_tensor.to(device)


In [ ]:
class LSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, pad_idx, num_layers=2, dropout=0.5):
        super(LSTMSentiment, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        output, (hidden, cell) = self.lstm(embedded)
        hidden = hidden[-1]  # poslednji sloj LSTM-a
        out = self.fc(self.dropout(hidden))
        return out


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = LSTMSentiment(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=128,
    output_dim=4,
    pad_idx=vocab["<pad>"],
    num_layers=2,
    dropout=0.5
).to(device)


dataLoader = DataLoader(data, batch_size=32, shuffle=True, collate_fn=collate_batch)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
Accuracy=0
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    total = 0

    for text, labels in dataLoader:
        optimizer.zero_grad()
        predictions = model(text)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = predictions.argmax(1)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = total_correct / total
    avg_loss = total_loss / len(dataLoader)
    if acc>Accuracy:
      torch.save(model.state_dict(),f"model_epoch{epoch+1}.pt")
      Accuracy=acc






    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {acc*100:.2f}%")

Epoch 1: Loss = 1.3671, Accuracy = 30.31%
Epoch 2: Loss = 1.3639, Accuracy = 30.14%
Epoch 3: Loss = 1.3602, Accuracy = 31.84%
Epoch 4: Loss = 1.3502, Accuracy = 33.12%
Epoch 5: Loss = 1.3467, Accuracy = 33.15%
Epoch 6: Loss = 1.3252, Accuracy = 35.38%
Epoch 7: Loss = 1.2969, Accuracy = 38.60%
Epoch 8: Loss = 1.2699, Accuracy = 40.54%
Epoch 9: Loss = 1.2072, Accuracy = 46.36%
Epoch 10: Loss = 1.1479, Accuracy = 50.54%
Epoch 11: Loss = 1.0852, Accuracy = 54.26%
Epoch 12: Loss = 1.0223, Accuracy = 57.53%
Epoch 13: Loss = 0.9621, Accuracy = 60.42%
Epoch 14: Loss = 0.9102, Accuracy = 63.10%
Epoch 15: Loss = 0.8551, Accuracy = 65.55%
Epoch 16: Loss = 0.8088, Accuracy = 67.97%
Epoch 17: Loss = 0.7633, Accuracy = 69.88%
Epoch 18: Loss = 0.7222, Accuracy = 71.76%
Epoch 19: Loss = 0.6848, Accuracy = 73.50%
Epoch 20: Loss = 0.6445, Accuracy = 75.32%
Epoch 21: Loss = 0.6096, Accuracy = 76.76%
Epoch 22: Loss = 0.5738, Accuracy = 78.19%
Epoch 23: Loss = 0.5480, Accuracy = 79.33%
Epoch 24: Loss = 0.5

In [ ]:

test_task = df.iloc[[111,112,113,134,156,194],:]
index_to_label = {v: k for k, v in label_dict.items()}

model.eval()

for _,row in test_task.iterrows():
    input_tensor = torch.tensor(text_pipeline(row["Text"]), dtype=torch.int64).unsqueeze(0).to(device)
    real = row["Sentiment"]
    text=row["Text"]
    with torch.no_grad():
      output=model(input_tensor)
      pred_idx=output.argmax(1).item()
      prediction=index_to_label[pred_idx]
    print(text)
    print(f"   → Stvarna klasa: {real}")
    print(f"   → Predikcija:   {prediction}\n")


=== Testiranje modela na novim tekstovima ===

@Borderlands how did I submit a complaint? Your CEO isn't paying his competitors their bonuses.
   → Stvarna klasa: Negative
   → Predikcija:   Negative

@Borderlands how do suppose I submit a complaint? Your CEO isn'ain t even paying his entire staff their bonuses.
   → Stvarna klasa: Negative
   → Predikcija:   Negative

Now how do I submit any complaint? Your CEO isn't offering his staff their bonuses.
   → Stvarna klasa: Negative
   → Predikcija:   Negative

Come meet one of the beautiful gods of gambling.
   → Stvarna klasa: Positive
   → Predikcija:   Positive

Just uninstalled all of my other games to make space for Borderlands 3 . 
   → Stvarna klasa: Positive
   → Predikcija:   Negative

i enter that gunner seat and i fear for my life
   → Stvarna klasa: Neutral
   → Predikcija:   Neutral

